# Magenta RealTime 2 on Kaggle (2x T4)


Three-cell setup: install via the curl|bash script (system Python,
non-editable, pinned non-drifting CUDA stack), then generate in a
subprocess (no kernel restart needed), then play. Run on a Kaggle notebook
with the GPU accelerator enabled (2x T4).


## 1. Install (clones fork, pins deps, downloads + quantizes)


In [ ]:
# Idempotent; safe to re-run. To use your own fork: set MRT_REPO=... before
# the curl. Ignore pip 'dependency conflict' warnings for Kaggle's RAPIDS
# (numba<0.62 vs MRT's numba>=0.65) — they're harmless for running MRT.
!curl -fsSL https://raw.githubusercontent.com/ctunix/magenta-realtime/main/scripts/install_kaggle.sh | bash


## 2. Generate 8 s of audio (sharded across 2 GPUs if present)


In [ ]:
# Runs in a subprocess (python -m ...) so NO kernel restart is needed.
# JAX memory env vars are set BEFORE the Python process starts.
# --shard shards the 2.4B model across all local CUDA GPUs (falls back to 1 GPU).
!XLA_PYTHON_CLIENT_PREALLOCATE=false XLA_PYTHON_CLIENT_MEM_FRACTION=0.85 MAGENTA_HOME=/kaggle/working python3 -m magenta_rt.jax.generate --model mrt2_base --checkpoint mrt2_base_bf16.safetensors --shard --duration 8 --prompt "disco funk"


## 3. Play the result


In [ ]:
import IPython.display as ipd
ipd.display(ipd.Audio('/kaggle/working/magenta-rt-v2/outputs/output_audio_jax_mrt2_base.wav', rate=48000))


## Notes


- `MAGENTA_HOME` is the **base** dir (`/kaggle/working`); `paths.py` appends
`magenta-rt-v2` itself. On Colab, use `/content` instead of `/kaggle/working`.
- To keep the fp32 checkpoint (for fp32 + 2-GPU sharding), re-run with
`KEEP_FP32=1 curl ... | bash` and use `--checkpoint mrt2_base.safetensors`.
- To change the prompt/duration, edit the `--prompt`/`--duration` in step 2.
